In [1]:
# !pip install --upgrade transformers #pip install --upgrade transformers

In [2]:
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM

In [3]:
import torch

In [ ]:
model_id = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id,device_map='cuda',dtype = torch.bfloat16)

In [ ]:
pipe = pipeline(task='text-generation',model=model,tokenizer=tokenizer,return_full_text=False,temperature=0.5,top_k=20,top_p=0.95,max_new_tokens=300)

In [ ]:
prompts
#'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n[query]<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [11]:
query = 'who are you?'
def generate_text(query):
  messages = [
    {'role':'system', 'content':'You are a helpful assistant.'},
    {'role':'user', 'content':'[query]'}
]
  prompts = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
  prompts = prompts.replace('[query]',query)
  with torch.no_grad():
    set_seed(42)
    return pipe(prompts)[0]['generated_text']

In [ ]:
res = generate_text(query)

In [ ]:
print(res)

output: </br>
I'm Qwen3.5, the latest large language model developed by Tongyi Lab. I'm designed to assist with a wide range of tasks, from answering questions and writing stories to coding, logical reasoning, and analyzing complex data. How can I help you?

In [20]:
tool_usage = """
{'calculation':'Used to calculate or to do any mathematical operations. It includes addition, substraction, multiplication, division. It also finds out square root and square of a number.'
'search':'it is used to search external website when internal information is not sufficient or model does not have information.',
'from_sql':'Extract the data from sql'}
"""
system_prompt=f"""
You are a helpful assistant who is excellent at tool calling. Your task is to identify which tools need to be used.
You will be provided with along with thier usage.
{tool_usage}
provide your response in json format.
"""

In [21]:
def generate_text(query,system_prompt):
  messages = [
    {'role':'system', 'content':system_prompt},
    {'role':'user', 'content':'[query]'}
]
  prompts = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
  prompts = prompts.replace('[query]',query)
  with torch.no_grad():
    set_seed(42)
    return pipe(prompts)[0]['generated_text']

In [ ]:
query="what is the weather for new delhi today?"
res=generate_text(query,system_prompt)
res

In [ ]:
print(res)

output: </br>
```python
{
  "tool": "search",
  "description": "Search external websites to find the current weather in New Delhi."
}
```

In [ ]:
query="i need to calculate 2+3."
res=generate_text(query,system_prompt)
print(res)

output: </br>
```python
{
  "tool": "calculation",
  "arguments": {
    "operation": "addition",
    "operands": [2, 3]
  }
}
```

### Test pandas

In [25]:
import pandas as pd

In [ ]:
df = pd.read_excel('call_center_data.xlsx')
df.head()

In [ ]:
df.info()

In [35]:
information = """
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   ID                       70 non-null     object
 1   Customer Name            70 non-null     object
 2   Sentiment                70 non-null     object
 3   CSAT Score               70 non-null     int64
 4   Call Timestamp           70 non-null     datetime64[ns]
 5   Reason                   70 non-null     object
 6   City                     70 non-null     object
 7   State                    70 non-null     object
 8   Channel                  70 non-null     object
 9   Response Time            70 non-null     object
 10  Call Duration (Minutes)  70 non-null     int64
 11  Call Center              70 non-null     object
dtypes: datetime64[ns](1), int64(2), object(9)
memory usage: 6.7+ KB
"""
columns = """
['ID', 'Customer Name', 'Sentiment', 'CSAT Score', 'Call Timestamp',
       'Reason', 'City', 'State', 'Channel', 'Response Time',
       'Call Duration (Minutes)', 'Call Center']
"""
head = """
ID	Customer Name	Sentiment	CSAT Score	Call Timestamp	Reason	City	State	Channel	Response Time	Call Duration (Minutes)	Call Center
0	PIS-39858047	Frances Gullefant	Neutral	8	2024-01-01	Billing Question	Charleston	West Virginia	Chatbot	Within SLA	14	Chicago/IL
1	ROH-40287216	Harcourt Zoanetti	Negative	5	2024-01-01	Billing Question	Honolulu	Hawaii	Call-Center	Within SLA	32	Los Angeles/CA
2	NKC-39578431	Washington Oxteby	Negative	3	2024-01-02	Billing Question	Dallas	Texas	Chatbot	Within SLA	6	Denver/CO
"""

In [38]:
system_prompt=f""" You are an AI assistant who is excellent at writing python pandas code. Based on user query, write the pandas code.
you will be provided with columns and informations for that dataframe.
Here are columns for the dataset - {columns} and informations related to dataset - {information} and top 3 rows of the dataset - {head}.
Return only pandas code.
"""

In [39]:
def generate_text(query,system_prompt=system_prompt):
  messages = [
    {'role':'system', 'content':system_prompt},
    {'role':'user', 'content':'[query]'}
]
  prompts = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
  prompts = prompts.replace('[query]',query)
  with torch.no_grad():
    set_seed(42)
    return pipe(prompts)[0]['generated_text']

In [ ]:
query = "Find out sentiments for the call center located in 'Los Angeles/CA' along with the counts of sentiment in descending order."
res =generate_text(query)
print(res)

outputs: </br>
```python
# Filter the dataframe for the 'Los Angeles/CA' call center
la_df = df[df['Call Center'] == 'Los Angeles/CA']

# Group by 'Sentiment' and count occurrences
sentiment_counts = la_df.groupby('Sentiment').size()

# Sort the counts in descending order
sentiment_counts_sorted = sentiment_counts.sort_values(ascending=False)

# Display the result
print(sentiment_counts_sorted)
```

In [ ]:
exec(res.replace('```python','').replace("```",""))

In [ ]:
df[df['Call Center']=='Los Angeles/CA']['Sentiment'].value_counts().sort_values(ascending=False)

In [ ]:
query = "Draw a pie chart based on sentiments for the call center located in 'Los Angeles/CA' along with the counts of sentiment."
res =generate_text(query)
print(res)

Outputs: </br>
```python
import pandas as pd
import matplotlib.pyplot as plt

# Assuming 'df' is the DataFrame variable containing the data
df_los_angeles = df[df['Call Center'] == 'Los Angeles/CA']

# Count sentiments for the selected call center
sentiment_counts = df_los_angeles['Sentiment'].value_counts()

# Create the pie chart
plt.figure(figsize=(8, 6))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Sentiment Distribution for Call Center: Los Angeles/CA')
plt.axis('equal')  # Equal aspect ratio ensures the pie chart is circular
plt.show()
```

In [ ]:
exec(res.replace('```python','').replace("```",""))

In [ ]:
query = "show me sentiments distribution for the first 10 days."
res =generate_text(query)
print(res)

outputs: </br>
```python
import pandas as pd

# Assuming the dataframe is named 'df'
# Filter for the first 10 days based on 'Call Timestamp'
df_first_10_days = df[df['Call Timestamp'] <= pd.Timestamp('2024-01-10')]

# Calculate sentiments distribution
sentiment_distribution = df_first_10_days['Sentiment'].value_counts()

# Display the result
print(sentiment_distribution)
```

In [ ]:
exec(res.replace('```python','').replace("```",""))

In [ ]:
df_first_10_days = df[df['Call Timestamp'] <= pd.Timestamp('2024-01-10')]

# Calculate sentiments distribution
sentiment_distribution = df_first_10_days['Sentiment'].value_counts()

# Display the result
print(sentiment_distribution)

In [ ]:
df_first_10_days

In [ ]:
df.head(10)